# Format Deep Dive: **Lance** — the ML-native columnar format

*Part of the weyland notebook library (B81). Self-contained: it builds its own
sample data — including a random embedding column — writes a real Lance dataset
to a temp directory, and runs every step top-to-bottom. No lakeFS, no MinIO, no
network.*

---

## What is Lance?

[**Lance**](https://lancedb.github.io/lance/) is a modern **columnar format**
purpose-built for machine-learning data. Think of it as *"Parquet, redesigned
for the workloads ML actually has"*:

| Capability | What Lance gives you |
|---|---|
| **Fast random access** | O(1)-ish point lookups by row id — `dataset.take([...])` fetches specific rows **without scanning the whole file**. Parquet has to walk row groups. |
| **Zero-copy versioning** | Every write is a new *version*. Old versions stay valid and cost nothing extra — Lance writes new data files and a tiny new manifest; it never rewrites what didn't change. Time-travel is a `version=N` read away. |
| **First-class vector index** | Embeddings are a native column type (`fixed_size_list<float>`), and Lance builds an **ANN index** (IVF_PQ, IVF_HNSW, …) *inside the dataset* — vector search is a query on the table, not a separate service. |
| **Mutation** | Append, overwrite, delete, and update in place — Parquet files are immutable. |
| **Directory dataset** | A `.lance` dataset is a **directory**: `data/` fragments + a `_versions/` manifest log + `_indices/`. Self-describing, works on local disk or object storage. |

### Lance vs. Parquet — the one-line contrast

> **Parquet** is optimized for *immutable, full-column scans* — write once, read
> the whole column fast. It has **no random access and no vector index**.
>
> **Lance** is optimized for *point lookups, vector search, and mutation* — the
> access pattern of a feature store or a RAG retriever, where you grab **row
> 47,213 and its 128-d embedding's 10 nearest neighbours**, not "all of
> column X".

Both are columnar and both are great analytical scanners. You reach for Lance
when the *access pattern* is random + vector + versioned, and for Parquet when
it's a wide immutable scan feeding a query engine.

This notebook walks the four things Lance does that Parquet can't:
**versioning / time-travel**, **cheap random access**, and **a built-in vector
index for ANN search** — plus how it reads back into pandas / polars / DuckDB.


## 0. Setup — libraries and a scratch directory

Everything is local. We write the dataset under a temp directory that is cleaned
up at the end, so this notebook leaves nothing behind and can be re-run any number
of times.

In [1]:
import os
import shutil
import tempfile

import numpy as np
import pyarrow as pa
import pandas as pd
import polars as pl
import duckdb

import lance  # pylance; the package installs as `pylance`, imported as `lance`

print("lance   :", lance.__version__)
print("pyarrow :", pa.__version__)
print("polars  :", pl.__version__)
print("duckdb  :", duckdb.__version__)
print("numpy   :", np.__version__)

# Scratch dir for the dataset (a directory-based dataset lives here).
WORKDIR = tempfile.mkdtemp(prefix="lance_demo_")
DS_PATH = os.path.join(WORKDIR, "movies.lance")
print("\nDataset path:", DS_PATH)

lance   : 11.0.0
pyarrow : 25.0.0
polars  : 1.44.1
duckdb  : 1.5.5
numpy   : 2.4.6

Dataset path: /tmp/lance_demo_k1o3rl15/movies.lance


## 1. Build sample data *in the notebook* — with an embedding column

We fabricate a tiny "movies" catalog: a few hundred rows with an id, a title, a
genre, a numeric rating, and — the part that matters for ML — a **16-dimensional
`float32` embedding** per row (a stand-in for whatever a real encoder would emit).

The embedding column is a PyArrow `fixed_size_list<float32>[16]`. That fixed-size
list is exactly the type Lance recognises as a vector column and can build an ANN
index on.

In [2]:
N_ROWS = 400
DIM = 16
rng = np.random.default_rng(42)

genres = np.array(["scifi", "drama", "comedy", "horror", "doc"])
# Give each genre its own cluster centre in embedding space so that
# nearest-neighbour search returns something intuitively sensible later.
genre_centres = rng.standard_normal((len(genres), DIM)).astype(np.float32) * 3.0

row_genre_idx = rng.integers(0, len(genres), size=N_ROWS)
# embedding = genre centre + small per-row noise -> tight, separable clusters
embeddings = (
    genre_centres[row_genre_idx] + rng.standard_normal((N_ROWS, DIM)).astype(np.float32)
).astype(np.float32)

ids = np.arange(N_ROWS, dtype=np.int64)
titles = [f"Title #{i:04d}" for i in range(N_ROWS)]
genre_vals = genres[row_genre_idx]
ratings = np.round(rng.uniform(1.0, 10.0, size=N_ROWS), 1).astype(np.float32)

# A fixed_size_list<float32>[DIM] column is what Lance treats as a vector.
embedding_col = pa.FixedSizeListArray.from_arrays(
    pa.array(embeddings.reshape(-1), type=pa.float32()), DIM
)

table = pa.table(
    {
        "id": pa.array(ids, pa.int64()),
        "title": pa.array(titles, pa.string()),
        "genre": pa.array(genre_vals, pa.string()),
        "rating": pa.array(ratings, pa.float32()),
        "embedding": embedding_col,
    }
)

print(f"{N_ROWS} rows, embedding dim = {DIM}")
print(table.schema)
table.slice(0, 3).to_pandas()

400 rows, embedding dim = 16
id: int64
title: string
genre: string
rating: float
embedding: fixed_size_list<item: float>[16]
  child 0, item: float


,id,title,genre,rating,embedding
0,0,Title #0000,comedy,9.9,"[1.2773557, -0.34815, 3.4212933, 3.7727635, -1..."
1,1,Title #0001,doc,7.5,"[1.8415086, 2.2765534, -1.2981527, -1.2344929,..."
2,2,Title #0002,drama,4.1,"[1.044928, -3.9295464, 2.3008945, 1.1502668, 0..."


## 2. Write a Lance dataset, read it back, inspect it

`lance.write_dataset(table, path)` materialises a **directory-based dataset**.
Let's write it, reopen it with `lance.dataset(path)`, and look at both the schema
and the on-disk layout.

In [3]:
# mode="create" writes a fresh dataset (this is version 1).
lance.write_dataset(table, DS_PATH, mode="create")

ds = lance.dataset(DS_PATH)
print("rows      :", ds.count_rows())
print("version   :", ds.version)
print("schema    :")
print(ds.schema)

rows      : 400
version   : 1
schema    :
id: int64
title: string
genre: string
rating: float
embedding: fixed_size_list<item: float>[16]
  child 0, item: float


In [4]:
# A .lance "file" is really a directory. This is what makes versioning and
# zero-copy time-travel possible: data fragments, a manifest log, indices.
def show_tree(root, prefix=""):
    entries = sorted(os.listdir(root))
    for name in entries:
        full = os.path.join(root, name)
        marker = "/" if os.path.isdir(full) else ""
        print(f"{prefix}{name}{marker}")
        if os.path.isdir(full):
            show_tree(full, prefix + "    ")

print(os.path.basename(DS_PATH), "(directory dataset)")
show_tree(DS_PATH, "    ")

movies.lance (directory dataset)
    _transactions/
        0-8d4b9bc5-53f5-428c-a477-f154de1ac3b2.txn
    _versions/
        18446744073709551614.manifest
        latest_version_hint.json
    data/
        11010110110011100110111090185c43fd861023ef03ceab3a.lance


In [5]:
# Read the whole thing back into the analytical engines it interoperates with.
# pandas:
df = ds.to_table().to_pandas()
print("pandas  ->", df.shape)

# polars (zero-copy from the Arrow table):
pdf = pl.from_arrow(ds.to_table())
print("polars  ->", pdf.shape)

# DuckDB can query the Arrow table directly (no copy, no registration ceremony):
arrow_tbl = ds.to_table()
avg_by_genre = duckdb.sql(
    "SELECT genre, COUNT(*) AS n, ROUND(AVG(rating), 2) AS avg_rating "
    "FROM arrow_tbl GROUP BY genre ORDER BY n DESC"
).to_df()
print("\nDuckDB aggregate over the Lance-backed Arrow table:")
avg_by_genre

pandas  -> (400, 5)
polars  -> (400, 5)

DuckDB aggregate over the Lance-backed Arrow table:


,genre,n,avg_rating
0,scifi,93,5.58
1,comedy,83,5.38
2,doc,80,5.68
3,drama,73,5.16
4,horror,71,5.69


## 3. Versioning & time-travel — *zero-copy*

This is the headline feature. Every write to a Lance dataset produces a new
**version**. Crucially it is **zero-copy**:

- A new version writes **only the new/changed data fragments** plus a tiny new
  **manifest** (a pointer file in `_versions/`) that lists which fragments make
  up that version.
- Unchanged fragments are **shared** across versions — nothing is duplicated,
  nothing is rewritten.
- Old versions therefore remain fully readable **for free**. "Time-travel" is
  just opening the dataset pinned to an older manifest.

Let's create version 2 by **appending** 100 more rows, then a version 3 by an
**overwrite**, then travel back.

In [6]:
# --- Version 2: APPEND 100 more rows (ids 400..499) ---
N_MORE = 100
more_genre_idx = rng.integers(0, len(genres), size=N_MORE)
more_emb = (
    genre_centres[more_genre_idx] + rng.standard_normal((N_MORE, DIM)).astype(np.float32)
).astype(np.float32)
more = pa.table(
    {
        "id": pa.array(np.arange(N_ROWS, N_ROWS + N_MORE, dtype=np.int64)),
        "title": pa.array([f"Title #{i:04d}" for i in range(N_ROWS, N_ROWS + N_MORE)]),
        "genre": pa.array(genres[more_genre_idx]),
        "rating": pa.array(np.round(rng.uniform(1, 10, N_MORE), 1).astype(np.float32)),
        "embedding": pa.FixedSizeListArray.from_arrays(
            pa.array(more_emb.reshape(-1), pa.float32()), DIM
        ),
    }
)
lance.write_dataset(more, DS_PATH, mode="append")

ds = lance.dataset(DS_PATH)
print("after append -> version", ds.version, "rows", ds.count_rows())

after append -> version 2 rows 500


In [7]:
# --- Version 3: OVERWRITE with just the first 50 rows ---
# `overwrite` starts a new logical table state but does NOT delete the old data
# files -- the previous versions still point at them, so time-travel still works.
lance.write_dataset(table.slice(0, 50), DS_PATH, mode="overwrite")

ds = lance.dataset(DS_PATH)
print("after overwrite -> version", ds.version, "rows", ds.count_rows())

after overwrite -> version 3 rows 50


In [8]:
# List every version. Each entry is a manifest snapshot with a timestamp.
versions = ds.versions()
print(f"{len(versions)} versions:\n")
for v in versions:
    ts = v.get("timestamp")
    print(f"  version {v['version']}  @ {ts}")

3 versions:

  version 1  @ 2026-09-01 13:07:36.333118
  version 2  @ 2026-09-01 13:07:36.356035
  version 3  @ 2026-09-01 13:07:36.359830


In [9]:
# Time-travel: open the dataset AS IT WAS at each version and count rows.
# This is a cheap metadata read -- it just resolves a different manifest.
print("Row count at each version (time-travel):")
for v in versions:
    n = v["version"]
    past = lance.dataset(DS_PATH, version=n)   # pin to an old manifest
    print(f"  version {n}: {past.count_rows():>4} rows")

# checkout_version does the same thing from an already-open handle:
v1 = ds.checkout_version(1)
print("\ncheckout_version(1) row count:", v1.count_rows(),
      "-- original 400 rows are still fully readable, zero-copy.")

# And restore/back to the tip:
ds = lance.dataset(DS_PATH)   # latest
print("latest version:", ds.version, "rows", ds.count_rows())

Row count at each version (time-travel):
  version 1:  400 rows
  version 2:  500 rows
  version 3:   50 rows

checkout_version(1) row count: 400 -- original 400 rows are still fully readable, zero-copy.
latest version: 3 rows 50


## 4. Fast random access — `.take([row_ids])`

A feature store rarely wants "all of column X". It wants *these specific rows* —
the 256 examples in this training batch, the 10 candidates a retriever just
ranked. Lance is built for that.

`dataset.take([i, j, k, ...])` fetches **exactly those rows by position** and
returns an Arrow table. Why it's cheap in Lance and painful in Parquet:

- Lance stores per-column data with structure that lets it **seek directly to a
  row's location** and read just that value — random reads don't drag in a whole
  row group.
- Parquet's unit of access is the **row group** (often 10k–1M rows); to read one
  row you decode the whole group its column chunk lives in. Point lookups are an
  antipattern there.

So `.take` turns "give me rows 5, 100, and 399" into three small seeks, not a
scan.

In [10]:
# Reopen at the latest version that still has all rows (version 2 had 500).
ds_full = lance.dataset(DS_PATH, version=2)
print("using version 2 with", ds_full.count_rows(), "rows for the take() demo\n")

wanted = [5, 100, 399, 450, 499]           # arbitrary, non-contiguous row ids
picked = ds_full.take(wanted, columns=["id", "title", "genre", "rating"])
print("take() fetched", picked.num_rows, "specific rows without a full scan:")
picked.to_pandas()

using version 2 with 500 rows for the take() demo

take() fetched 5 specific rows without a full scan:


,id,title,genre,rating
0,5,Title #0005,scifi,3.0
1,100,Title #0100,doc,5.4
2,399,Title #0399,scifi,6.1
3,450,Title #0450,doc,8.4
4,499,Title #0499,drama,9.9


In [11]:
# You can take the embedding too -- e.g. to hydrate the vectors for a
# specific training batch by row id.
batch_ids = [0, 1, 2, 250, 480]
vecs = ds_full.take(batch_ids, columns=["id", "embedding"]).to_pandas()
mat = np.vstack(vecs["embedding"].to_numpy())
print("hydrated", mat.shape[0], "embeddings of dim", mat.shape[1],
      "by row id -- no scan of the other ~495 rows.")
vecs[["id"]].assign(emb_first3=[e[:3] for e in vecs["embedding"]])

hydrated 5 embeddings of dim 16 by row id -- no scan of the other ~495 rows.


,id,emb_first3
0,0,"[1.2773557, -0.34815, 3.4212933]"
1,1,"[1.8415086, 2.2765534, -1.2981527]"
2,2,"[1.044928, -3.9295464, 2.3008945]"
3,250,"[1.3087507, -1.2758358, 2.6450787]"
4,480,"[-0.24495173, -2.6364405, 1.8654757]"


## 5. Vector index + ANN search

Now the ML headline: Lance builds an **approximate-nearest-neighbour (ANN)
index** *inside the dataset*, so vector search is a query on the table — no
separate vector service, no second copy of the data.

We build an **IVF_PQ** index on the `embedding` column:

- **IVF** (*Inverted File*) partitions the vectors into `num_partitions`
  Voronoi cells; a query only scans the few nearest cells instead of all
  vectors.
- **PQ** (*Product Quantization*) compresses each vector into
  `num_sub_vectors` codes, shrinking the index and speeding distance math.

For our few-hundred-row toy set we use **small** params (`num_partitions=4`,
`num_sub_vectors=4` — 4 evenly divides our dim of 16). On a real corpus of
millions you'd use hundreds/thousands of partitions.

An index is committed against the dataset's **latest** version, so we first
`restore()` the 500-row version 2 back to the tip (§3 left the tip at the 50-row
overwrite). `restore` is itself zero-copy — it writes a new manifest that points
at version 2's existing fragments; it copies no data.

> **Robustness note:** IVF_PQ trains a k-means model, which needs enough rows.
> The cell below **attempts the real IVF_PQ index** and, if training ever fails
> on a too-small set, falls back cleanly. **Either way the ANN query in the next
> cell runs** — Lance's `nearest=` performs an exact **brute-force** kNN when no
> index is present, so the search is always demonstrated.

In [12]:
# An index commits against the LATEST version. §3 left the tip at the 50-row
# overwrite (v3), so restore the full 500-row v2 to the tip first (zero-copy:
# it just writes a new manifest pointing at v2's existing fragments).
ds_vec = lance.dataset(DS_PATH).checkout_version(2)
ds_vec.restore()
ds_vec = lance.dataset(DS_PATH)   # now the tip, 500 rows
print("indexing against version", ds_vec.version, "with", ds_vec.count_rows(), "rows\n")

INDEX_KIND = None
try:
    ds_vec.create_index(
        column="embedding",
        index_type="IVF_PQ",
        metric="L2",
        num_partitions=4,     # small -> few Voronoi cells for a tiny corpus
        num_sub_vectors=4,    # 4 divides dim=16 -> 4-dim sub-quantizers
        replace=True,
    )
    INDEX_KIND = "IVF_PQ (real ANN index)"
    print("Built IVF_PQ index:")
    for ix in ds_vec.list_indices():
        print("  ", {k: ix[k] for k in ("name", "type", "fields")})
except Exception as e:  # pragma: no cover - fallback path for tiny corpora
    INDEX_KIND = "brute-force (index training failed, using flat kNN)"
    print("IVF_PQ training failed:", type(e).__name__, str(e)[:200])
    print("-> falling back to brute-force nearest-neighbour search.")

print("\nSearch strategy:", INDEX_KIND)

indexing against version 4 with 500 rows

Built IVF_PQ index:
   {'name': 'embedding_idx', 'type': 'IVF_PQ', 'fields': ['embedding']}

Search strategy: IVF_PQ (real ANN index)


In [13]:
# Run a k-NN query. `nearest=` uses the index if one exists, otherwise it does
# an exact brute-force scan -- so this cell works with OR without the index.
K = 5

# Query vector: sit it right on the "scifi" cluster centre so the nearest
# neighbours should mostly be scifi rows -> an intuitive, checkable result.
query_vec = genre_centres[0].astype(np.float32)   # genres[0] == "scifi"

knn = ds_vec.to_table(
    nearest={
        "column": "embedding",
        "q": query_vec,
        "k": K,
        # nprobes only matters for the IVF index; harmless for brute force.
        "nprobes": 4,
    },
    columns=["id", "title", "genre", "rating"],
).to_pandas()

print(f"Top-{K} nearest rows to a 'scifi'-centred query vector")
print(f"(via {INDEX_KIND}):\n")
print(knn.to_string(index=False))
print("\n_distance is L2 (smaller = closer). Expect mostly genre == 'scifi'.")

Top-5 nearest rows to a 'scifi'-centred query vector
(via IVF_PQ (real ANN index)):

 id       title genre  rating  _distance
332 Title #0332 scifi     6.8   4.977027
163 Title #0163 scifi     9.0   5.634429
 13 Title #0013 scifi     2.3   6.723387
397 Title #0397 scifi     5.3   6.834350
221 Title #0221 scifi     3.5   6.887073

_distance is L2 (smaller = closer). Expect mostly genre == 'scifi'.


[2026-09-01T17:07:36Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_distance`.  Currently the `_distance` column will be included.  In the future it will not.  Call `disable_scoring_autoprojection` to adopt the future behavior and avoid this warning


In [14]:
# Sanity-check the ANN result against an exact numpy brute-force kNN.
# On this tightly-clustered toy set the ANN top-k should closely match exact.
all_rows = ds_vec.to_table(columns=["id", "genre", "embedding"]).to_pandas()
M = np.vstack(all_rows["embedding"].to_numpy())
d2 = np.sum((M - query_vec) ** 2, axis=1)          # squared L2 to every row
exact_order = np.argsort(d2)[:K]
exact = all_rows.iloc[exact_order][["id", "genre"]].assign(l2=np.sqrt(d2[exact_order]))
print("Exact numpy brute-force top-5 (ground truth):")
print(exact.to_string(index=False))
print("\nGenre of the query centre: 'scifi' ->",
      f"{(exact['genre'] == 'scifi').sum()}/5 exact neighbours are scifi.")

Exact numpy brute-force top-5 (ground truth):
 id genre       l2
332 scifi 2.097913
397 scifi 2.240004
157 scifi 2.563271
163 scifi 2.569108
481 scifi 2.627096

Genre of the query centre: 'scifi' -> 5/5 exact neighbours are scifi.


## 6. When to use Lance — and when not to

### Reach for **Lance** when the access pattern is ML-shaped
- **Vector search / RAG retrieval.** Embeddings + ANN index living *with* the
  data. This is exactly the job of the lab's dedicated vector stores —
  **Qdrant** and **Weaviate** (B78) — and of **LanceDB**, which is *built on
  this format*. Lance is the on-disk substrate; LanceDB is the serving layer over
  it. When you want vector search **without standing up a service** — a notebook,
  a batch job, an embedded retriever — a `.lance` directory *is* the database.
- **ML feature stores.** Point lookups by row id (`.take`) to hydrate a training
  batch or serve features, over millions of rows, cheaply.
- **Versioned datasets.** Zero-copy snapshots for reproducible training runs —
  pin `version=N` and you have the exact data a model saw, with no storage
  blow-up.
- **Mutable datasets.** Append/overwrite/delete/update in place, which immutable
  formats can't do.

### Reach for **Parquet / Arrow / Avro** instead when…
| Format | Best when |
|---|---|
| **Parquet** | Immutable, wide **columnar scans** feeding a query/analytics engine (Trino, DuckDB, Spark). The lakehouse table format (Iceberg / Delta on the lab's Nessie + lakeFS) is Parquet underneath. No random access or vectors needed. |
| **Arrow (IPC/Feather)** | In-memory / zero-copy **interchange** between processes and languages *right now*; not a long-term storage format. |
| **Avro** | **Row-oriented** streaming and message payloads (Kafka / Redpanda), schema evolution on the wire — write/read whole records, not columns. |

### The one-sentence rule of thumb
> If your reads are **"row 47,213 and its nearest neighbours,"** use **Lance**.
> If they're **"all of column X across a billion rows,"** use **Parquet**.
> If they're **"the next record off the topic,"** use **Avro**.

In the weyland platform Lance sits alongside — not instead of — the Iceberg
lakehouse: the lake holds the big immutable analytical tables (Parquet), while
Lance is the natural home for **embedding tables and feature/vector workloads**
that want random access, versioning, and a built-in ANN index.

In [15]:
# Cleanup -- remove the scratch dataset so the notebook leaves nothing behind.
shutil.rmtree(WORKDIR, ignore_errors=True)
print("Removed scratch dir:", WORKDIR)
print("Done. This notebook is self-contained and re-runnable.")

Removed scratch dir: /tmp/lance_demo_k1o3rl15
Done. This notebook is self-contained and re-runnable.
